In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd
import numpy as np
import os

from astropy.time import Time

In [ ]:
# set the directory path
directory_path = 'epoch_1/'

# create an empty list to store dataframes
dfs = []

# loop through all files in the directory that start with "beam_inf"
for filename in os.listdir(directory_path):
    if filename.startswith('beam_inf') and filename.endswith('.csv'):
        # read the CSV file into a pandas dataframe
        filepath = os.path.join(directory_path, filename)
        df = pd.read_csv(filepath)
        # append the dataframe to the list
        dfs.append(df)
        
# concatenate all dataframes into a single dataframe
combined_df = pd.concat(dfs, ignore_index=True)


In [ ]:
# plot the beam time of each dataframe
plt.figure(figsize=(20, 5))
plt.scatter(combined_df['BEAM_TIME'], combined_df['BEAM_NUM'])
plt.title('Beam Time')
plt.xlabel('Time')
plt.ylabel('Beam Number')
plt.show()

In [ ]:
# create an empty list to store the max and min values
max_min_values = []

# loop through all dataframes in the list
for df in dfs:
    # find the max and min values of RA_DEG and DEC_DEG
    max_ra = df['RA_DEG'].max()
    min_ra = df['RA_DEG'].min()
    max_dec = df['DEC_DEG'].max()
    min_dec = df['DEC_DEG'].min()
    diff_ra = max_ra - min_ra
    diff_dec = max_dec - min_dec
    # append the max and min values to the list
    max_min_values.append({'max_ra': max_ra, 'min_ra': min_ra, 'max_dec': max_dec, 'min_dec': min_dec,
                           'diff_ra': diff_ra, 'diff_dec': diff_dec})

# create a new dataframe with the max and min values
max_min_df = pd.DataFrame(max_min_values)
print(max_min_df)


In [ ]:
# create an empty list to store the max and min values
beam_time_values = []

# loop through all dataframes in the list
for df in dfs:
    # find the max and min values of RA_DEG and DEC_DEG
    max_time = df['BEAM_TIME'].max()
    min_time = df['BEAM_TIME'].min()
    diff_time = max_time - min_time
    # append the max and min values to the list
    beam_time_values.append({'max_time': max_time, 'min_time': min_time, 'diff_time': diff_time})

# create a new dataframe with the max and min values
beam_time_df = pd.DataFrame(beam_time_values)
print(beam_time_df)

In [ ]:
new_df = combined_df[combined_df['BEAM_TIME'].between(5.140e9, 5.145e9)]

plt.figure(figsize=(20, 5))
plt.scatter(new_df['BEAM_TIME'], new_df['BEAM_NUM'])
plt.title('Beam Time')
plt.xlabel('Time')
plt.ylabel('Beam Number')
plt.show()

In [ ]:
df_field_data = pd.read_csv('epoch_1/field_data.csv')
print(df_field_data['SCAN_START'])

In [ ]:
# remove all the invalid entries from the dataframe
df_field_data = df_field_data[df_field_data['SCAN_START'] != -1]
df_field_data = df_field_data[df_field_data['COMMENT'].isnull()]
df_field_data = df_field_data[df_field_data['SCAN_LEN'] > 800]


# plot the scan start time against the scan time length for each scan
plt.figure(figsize=(20, 5))
plt.scatter(df_field_data['SCAN_START'], df_field_data['SCAN_LEN'])
plt.title('Scan Time')
plt.xlabel('Scan Start Time')
plt.ylabel('Scan Time Length')
plt.show()

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_field_data['RA_DEG'], df_field_data['DEC_DEG'], c=df_field_data['SCAN_LEN'], cmap='jet', vmin=800)
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()


In [ ]:
def mjd2utc(mjd_seconds):
    # create a Time object with the MJD seconds
    t = Time(mjd_seconds/86400, format='mjd', scale='utc')

    # convert the time to YYYYMMDD HH:MM:SS format
    time_str = t.datetime.strftime('%Y-%m-%d %H:%M:%S')
    fin_time = time_str + '.' + str(mjd_seconds).split('.')[1]
    return fin_time


# For full RACSMid coverage

In [ ]:
# save the field names to a numpy array for use in the crossmatch notebook
field_list = df_field_data['FIELD_NAME'].tolist()
# np.save('RACSLow_Fields.npy', field_list)

sbid_list = df_field_data['SBID'].tolist()
# np.save('RACSLow_SBIDs.npy', sbid_list)

cal_sbid_list = df_field_data['CAL_SBID'].tolist()
# np.save('RACSLow_CAL_SBIDs.npy', cal_sbid_list)

df_field_data['UTC_SCAN_START'] = df_field_data['SCAN_START'].apply(mjd2utc)
time_list = df_field_data['UTC_SCAN_START'].tolist()
# np.save('RACSLow_Times.npy', time_list)

# print(field_list)

# Create a new dataframe
df_list = pd.DataFrame()

# Assign values to the columns
df_list['Field Name'] = field_list
df_list['SBID'] = sbid_list
df_list['CAL_SBID'] = cal_sbid_list
df_list['UTC Scan Start Time'] = time_list

df_list.sort_values('UTC Scan Start Time', inplace=True)
df_list.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
sbid = [str(df_list.iloc[i]['SBID']) for i in range(len(df_list))]
cal_sbid = [str(df_list.iloc[i]['CAL_SBID']) for i in range(len(df_list))]

cal_sbid_counts = Counter(cal_sbid)

cal_sbids = list(cal_sbid_counts.keys())
counts = list(cal_sbid_counts.values())

cal_sbid_to_sbid = {}

for i in range(len(df_list)):
    if cal_sbid[i] in cal_sbid_to_sbid:
        if sbid[i] not in cal_sbid_to_sbid[cal_sbid[i]]:
            cal_sbid_to_sbid[cal_sbid[i]].append(sbid[i])
    else:
        cal_sbid_to_sbid[cal_sbid[i]] = [sbid[i]]

# Print the corresponding SBID values for each CAL_SBID value
for cal_sbid_value, sbid_values in cal_sbid_to_sbid.items():
    print(f"CAL_SBID: {cal_sbid_value}, SBID values: {', '.join(sbid_values)}")

sbid_range = [f"{min(cal_sbid_to_sbid[cal_sbids[i]])}-{max(cal_sbid_to_sbid[cal_sbids[i]])}" for i in range(len(cal_sbids))]

plt.figure(figsize=(15, 6))
plt.bar(cal_sbids, counts)
plt.xlabel('SBID Range')
plt.ylabel('Number of Scans')
plt.title('SBID vs Number of Scans')
# Print the text at the bottom of the histogram
for i in range(len(sbid_range)):
    plt.text(i, counts[i], cal_sbids[i], ha='center', va='bottom', fontsize=8, rotation=90, mouseover=cal_sbids[i])

plt.xticks(range(len(sbid_range)), sbid_range, rotation=90)
plt.ylim(top=max(counts)+15)
plt.show()


In [ ]:
# Save the field names to a numpy array for use in the crossmatch notebook
# Convert df_list to a numpy array
df_list_array = np.array(df_list)

# Save the numpy array to a file
np.save('RACSMid_FullList.npy', df_list_array)


In [ ]:
date = [str(df_list.iloc[i]['UTC Scan Start Time'])[0:10] for i in range(len(df_list))]

date_counts = Counter(date)

for date, count in date_counts.items():
    print(f"{date}: {count}")

dates = list(date_counts.keys())
counts = list(date_counts.values())

plt.figure(figsize=(15, 6))
plt.bar(dates, counts)
plt.xlabel('Date')
plt.ylabel('Number of Scans')
plt.title('Date vs Number of Scans')
plt.xticks(rotation=90)
plt.show()

